In [1]:
import pandas as pd
from sqlalchemy import create_engine
import urllib.parse
from create_engine import ENGINE

query = """
SELECT *
FROM IDS_DATA_BEFORE_PREPROCESSING
"""

df = pd.read_sql(query, ENGINE)

print(df.tail())

2026-08-22 21:35:20,981 - INFO - Connection Built Successfully


Connecting to DB: localhost/cti_fyp_2
                                         flow id         src ip  src port  \
1058521   192.168.10.17-52.34.15.135-42319-443-6  192.168.10.17   42319.0   
1058522  192.168.10.14-52.87.129.147-50427-443-6  192.168.10.14   50427.0   
1058523    192.168.10.8-192.168.10.3-56351-53-17   192.168.10.8   56351.0   
1058524                                     None           None       NaN   
1058525    192.168.10.17-192.168.10.3-1411-53-17  192.168.10.17    1411.0   

                dst ip  dst port  protocol               timestamp  \
1058521   52.34.15.135     443.0         6  04/07/2017 03:23:51 PM   
1058522  52.87.129.147     443.0         6          5/7/2017 14:46   
1058523   192.168.10.3      53.0        17  04/07/2017 07:39:42 PM   
1058524           None       NaN        17                    None   
1058525   192.168.10.3      53.0        17  04/07/2017 06:32:44 PM   

         flow duration  total fwd packet  total bwd packets  ...  \
1058521   

In [2]:
drop_cols=["flow id", "src ip","dst ip", "src port", "dst port", "timestamp"]

In [3]:
# drop irrelevant cols
df_new = df.drop(drop_cols, axis=1)

# remove benign duplicates only
feature_cols = [c for c in df_new.columns if c != 'label']

benign = (
    df_new[df_new['label'] == 'BENIGN']
    .drop_duplicates(subset=feature_cols)
)

attack = df_new[df_new['label'] != 'BENIGN']

# combine
df_new = pd.concat([benign, attack], axis=0)

# reset index
df_new.reset_index(drop=True, inplace=True)

In [4]:
df_new.shape

(1035706, 78)

In [5]:
df_new.isnull().sum().sum()

np.int64(300)

In [6]:
df_new.duplicated().sum()

np.int64(158584)

In [46]:
1035145 - 158584

876561

In [7]:
df_new.dropna(inplace=True)

In [8]:
df_new.shape

(1035656, 78)

In [9]:
df_new['label'].value_counts()

label
BENIGN                          592822
PortScan                        159023
DoS Hulk                        158469
DDoS                             95123
DoS GoldenEye                     7567
DoS slowloris                     4001
FTP-Patator                       3973
DoS Slowhttptest - Attempted      3367
SSH-Patator                       2980
Bot                               2208
DoS Slowhttptest                  1742
DoS slowloris - Attempted         1706
Web Attack - Brute Force          1365
DoS Hulk - Attempted               579
Web Attack - XSS                   561
DoS GoldenEye - Attempted           80
Infiltration                        48
Web Attack - Sql Injection          12
Heartbleed                          11
FTP-Patator - Attempted             11
SSH-Patator - Attempted              8
Name: count, dtype: int64

In [10]:
# df_new.select_dtypes(include="object")
df_new["label"] = df_new["label"].str.replace('- Attempted', '', regex=False).str.strip()

In [11]:
df_new['label'].value_counts()

label
BENIGN                        592822
DoS Hulk                      159048
PortScan                      159023
DDoS                           95123
DoS GoldenEye                   7647
DoS slowloris                   5707
DoS Slowhttptest                5109
FTP-Patator                     3984
SSH-Patator                     2988
Bot                             2208
Web Attack - Brute Force        1365
Web Attack - XSS                 561
Infiltration                      48
Web Attack - Sql Injection        12
Heartbleed                        11
Name: count, dtype: int64

In [14]:
df_new.replace([np.inf, -np.inf], np.nan, inplace=True)

In [15]:
(df_new.select_dtypes(include='number') < 0).sum().sum()

np.int64(1195)

In [16]:
df_new.describe()

,protocol,flow duration,total fwd packet,total bwd packets,total length of fwd packet,total length of bwd packet,fwd packet length max,fwd packet length min,fwd packet length mean,fwd packet length std,...,fwd act data pkts,fwd seg size min,active mean,active std,active max,active min,idle mean,idle std,idle max,idle min
count,1.035656e+06,1.035656e+06,1.035656e+06,1.035656e+06,1.035656e+06,1.035656e+06,1.035656e+06,1.035656e+06,1.035656e+06,1.035656e+06,...,1.035656e+06,1.035656e+06,1.035656e+06,1.035656e+06,1.035656e+06,1.035656e+06,1.035656e+06,1.035656e+06,1.035656e+06,1.035656e+06
mean,9.900620e+00,1.099705e+07,1.139626e+01,1.257770e+01,5.136852e+02,1.966351e+04,2.005783e+02,1.594583e+01,4.444300e+01,6.139993e+01,...,1.970714e+00,1.959551e+01,1.453589e+05,5.769394e+04,2.350658e+05,1.112085e+05,4.078514e+06,3.060308e+05,4.359050e+06,3.793199e+06
std,5.267969e+00,2.880453e+07,7.816773e+02,1.043277e+03,6.279807e+03,2.353746e+06,4.692781e+02,3.263929e+01,9.201530e+01,1.508987e+02,...,1.129309e+01,1.054406e+01,8.201040e+05,4.763915e+05,1.206774e+06,7.393802e+05,1.311112e+07,2.837264e+06,1.385762e+07,1.284078e+07
min,0.000000e+00,-1.300000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,6.000000e+00,2.760000e+02,1.000000e+00,1.000000e+00,2.000000e+01,9.400000e+01,2.000000e+01,0.000000e+00,2.500000e+00,0.000000e+00,...,0.000000e+00,8.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,6.000000e+00,9.117400e+04,4.000000e+00,2.000000e+00,7.600000e+01,2.640000e+02,4.400000e+01,0.000000e+00,3.887500e+01,0.000000e+00,...,1.000000e+00,2.000000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,1.700000e+01,4.727466e+06,8.000000e+00,6.000000e+00,3.540000e+02,1.159500e+04,3.280000e+02,3.600000e+01,5.000000e+01,1.118775e+02,...,1.000000e+00,2.800000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
max,1.700000e+01,1.200000e+08,2.079630e+05,2.846030e+05,2.866077e+06,6.270390e+08,2.482000e+04,1.472000e+03,5.775500e+03,7.018511e+03,...,5.520000e+03,4.400000e+01,1.100975e+08,7.420000e+07,1.100975e+08,1.100975e+08,1.200000e+08,7.690000e+07,1.200000e+08,1.200000e+08


In [17]:
num_cols = df_new.select_dtypes(include=[np.number]).columns
neg_cols = (df_new[num_cols] > 0).sum()

In [18]:
neg_cols.sum()

np.int64(45448572)

In [19]:
iat_columns = [
        'flow iat min',
        'flow iat max', 
        'flow iat mean',
        'fwd iat min',
        'bwd iat min',
        'flow duration',
        'flow packets/s' 
    ]
    
for col in iat_columns:
    if col in df_new.columns:
            # Count negatives
        neg_count = (df_new[col] < 0).sum()
            
        if neg_count > 0:
                print(f"⚠️ Found {neg_count} negative values in {col}")
                
                # Check severity
                severe = (df_new[col] < -1).sum()
                
                if severe > 0:
                    print(f"  🔴 {severe} values < -1 (serious corruption)")
                
                # Fix: Replace negative with 0
                    df_new.loc[df_new[col] < 0, col] = 0
                    print(f"  ✅ Replaced with 0")

⚠️ Found 1175 negative values in flow iat min
  🔴 64 values < -1 (serious corruption)
  ✅ Replaced with 0
⚠️ Found 5 negative values in flow iat max
  🔴 3 values < -1 (serious corruption)
  ✅ Replaced with 0
⚠️ Found 5 negative values in flow iat mean
  🔴 3 values < -1 (serious corruption)
  ✅ Replaced with 0
⚠️ Found 5 negative values in flow duration
  🔴 3 values < -1 (serious corruption)
  ✅ Replaced with 0
⚠️ Found 5 negative values in flow packets/s
  🔴 5 values < -1 (serious corruption)
  ✅ Replaced with 0


In [20]:
df_new.describe()

,protocol,flow duration,total fwd packet,total bwd packets,total length of fwd packet,total length of bwd packet,fwd packet length max,fwd packet length min,fwd packet length mean,fwd packet length std,...,fwd act data pkts,fwd seg size min,active mean,active std,active max,active min,idle mean,idle std,idle max,idle min
count,1.035656e+06,1.035656e+06,1.035656e+06,1.035656e+06,1.035656e+06,1.035656e+06,1.035656e+06,1.035656e+06,1.035656e+06,1.035656e+06,...,1.035656e+06,1.035656e+06,1.035656e+06,1.035656e+06,1.035656e+06,1.035656e+06,1.035656e+06,1.035656e+06,1.035656e+06,1.035656e+06
mean,9.900620e+00,1.099705e+07,1.139626e+01,1.257770e+01,5.136852e+02,1.966351e+04,2.005783e+02,1.594583e+01,4.444300e+01,6.139993e+01,...,1.970714e+00,1.959551e+01,1.453589e+05,5.769394e+04,2.350658e+05,1.112085e+05,4.078514e+06,3.060308e+05,4.359050e+06,3.793199e+06
std,5.267969e+00,2.880453e+07,7.816773e+02,1.043277e+03,6.279807e+03,2.353746e+06,4.692781e+02,3.263929e+01,9.201530e+01,1.508987e+02,...,1.129309e+01,1.054406e+01,8.201040e+05,4.763915e+05,1.206774e+06,7.393802e+05,1.311112e+07,2.837264e+06,1.385762e+07,1.284078e+07
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,6.000000e+00,2.760000e+02,1.000000e+00,1.000000e+00,2.000000e+01,9.400000e+01,2.000000e+01,0.000000e+00,2.500000e+00,0.000000e+00,...,0.000000e+00,8.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,6.000000e+00,9.117400e+04,4.000000e+00,2.000000e+00,7.600000e+01,2.640000e+02,4.400000e+01,0.000000e+00,3.887500e+01,0.000000e+00,...,1.000000e+00,2.000000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,1.700000e+01,4.727466e+06,8.000000e+00,6.000000e+00,3.540000e+02,1.159500e+04,3.280000e+02,3.600000e+01,5.000000e+01,1.118775e+02,...,1.000000e+00,2.800000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
max,1.700000e+01,1.200000e+08,2.079630e+05,2.846030e+05,2.866077e+06,6.270390e+08,2.482000e+04,1.472000e+03,5.775500e+03,7.018511e+03,...,5.520000e+03,4.400000e+01,1.100975e+08,7.420000e+07,1.100975e+08,1.100975e+08,1.200000e+08,7.690000e+07,1.200000e+08,1.200000e+08


In [21]:
from sklearn.preprocessing import LabelEncoder

In [22]:
le = LabelEncoder()
df_new['label'] = le.fit_transform(df_new['label'])

In [23]:
df_new["label"].value_counts()

label
0     592822
4     159048
10    159023
2      95123
3       7647
6       5707
5       5109
7       3984
11      2988
1       2208
12      1365
14       561
9         48
13        12
8         11
Name: count, dtype: int64

In [24]:
import joblib

In [25]:
encoder_saved = joblib.dump(le, 'preprocessed_encoder.joblib')

In [26]:
import logging

logging.basicConfig(
    level=logging.INFO,  
    format="%(asctime)s - %(levelname)s - %(message)s"
)

df_new.to_sql(
    name="IDS_DATA_AFTER_PREPROCESSING",
    con=ENGINE,
    if_exists="fail",
    index=False,
    chunksize=10000
)
logging.info(f"Data stored in mysql successfully............")

C:\Users\UMAR.TECH\AppData\Local\Temp\ipykernel_25244\3875907023.py:8: UserWarning: The provided table name 'IDS_DATA_AFTER_PREPROCESSING' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  df_new.to_sql(
2026-08-22 21:59:22,937 - INFO - Data stored in mysql successfully............
